In [1]:
import requests
import pandas as pd
import time
import re
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import asyncio
import pandas as pd
from googletrans import Translator

In [2]:
try:
    df = pd.read_csv('ru_cefr_short.csv')
    print("✅ Файл найден:")
except:
    print("❌ Файл не найден, используем тестовые данные:")
    test_data = {
        'fragment': [
            "Весной, летом и осенью почти каждую субботу он играет в футбол. У них в школе есть футбольная команда. А зимой он играет в хоккей. Ещё мы любим театр.",
            "Вчера я весь вечер повторял грамматику, учил слова. В контрольной работе я сделал 3 ошибки. Питер и Кен написали всё без ошибок. Преподаватель сказал, что они делают большие успехи.",
            "В такой ситуации особенно сложно работающим студентам, которые должны находить время и для учёбы, и для работы. Это нередко вызывает стресс.",
            "Как редкое животное они стоили очень дорого и являлись предметом роскоши. Археологи нашли останки этих животных в развалинах домов богатых людей четвёртого века нашей эры.",
            "Он увлёкся энтомологией ещё мальчиком и в детстве познакомился с основными учёными трудами в этой области.",
            "Так же радиация — это частица, которая летит на огромной скорости. Сама частица может быть почти любой: от атомного ядра до электрона или фотона."
        ],
        'textbook-assigned cefr level': [1, 2, 3, 4, 5, 6]
    }
    df = pd.DataFrame(test_data)

df

✅ Файл найден:


,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [3]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

In [4]:
texts_to_augment = []

for i in range(len(train_labels)):
  if train_labels[i] == 6:
    texts_to_augment.append(train_texts[i])


len(texts_to_augment)

120

In [5]:
async def main(lang, full_name_lang):
    print(f"Аугментируем {len(texts_to_augment)} текстов...")
    df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

    translator = Translator()

    translated_to = await translator.translate(texts_to_augment, src='ru', dest=lang)
    print(f"Переведено на {full_name_lang}")
    
    translated_to = await translator.translate([trans.text for trans in translated_to], src=lang, dest='ru')
    print("Переведено обратно на русский")

    for n in range(len(texts_to_augment)):
        print(f"\n{n}. Реальный текст:\n\t{texts_to_augment[n]}")
        print(f"Аугметированный текст:\n\t{translated_to[n].text}")

        new_row = pd.DataFrame({
            'text': [texts_to_augment[n]],
            'augmented-text': [translated_to[n].text]
        })
        df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    output_filename = f"c2_augmented_one_times_translated_{lang}.csv"
    df_pred.to_csv(output_filename, index=False, encoding='utf-8-sig')


    return df_pred

# корейский

In [6]:
df_pred = await main('ko', 'корейский')

Аугментируем 120 текстов...
Переведено на корейский
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38-3:54 На самом деле радиация сама по себе не заразна. Например, мощные дозы радиации от одного и того же изотопа часто используются для стерилизации фруктов и овощей и для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в свою систему беспилотного вождения, а Waymo также запустила приложение для Android для владельцев беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон»

# персидский

In [7]:
df_pred = await main('fa', 'персидский')

Аугментируем 120 текстов...
Переведено на персидский
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38-3:54 На самом деле радиация сама по себе не заразна. Например, мощные дозы радиации одних и тех же изотопов часто используются для стерилизации фруктов и овощей, а также для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в свою систему беспилотного вождения, а Waymo даже запустила приложение для Android для владельцев беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин с

# финский

In [8]:
df_pred = await main('fi', 'финский')

Аугментируем 120 текстов...
Переведено на финский
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38–3:54 На самом деле радиация не заразна. Например, высокие дозы радиации одних и тех же изотопов часто используются для стерилизации фруктов и овощей, а также для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в миллиард долларов в собственную автоматизированную систему вождения, а Waymo даже запустила приложение для Android для владельцев своих беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин

# сербский

In [9]:
df_pred = await main('sr', 'сербский')

Аугментируем 120 текстов...
Переведено на сербский
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38–3:54 На самом деле радиация сама по себе не заразна. Например, мощные дозы радиации одних и тех же изотопов часто используются для стерилизации фруктов и овощей, а также для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в собственную систему беспилотного вождения, а Waymo даже запустила приложение для Android для владельцев беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядюш

# турецкий

In [10]:
df_pred = await main('tr', 'турецкий')

Аугментируем 120 текстов...
Переведено на турецкий
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38–3:54 На самом деле радиация сама по себе не заразна. Например, сильные дозы радиации одних и тех же изотопов часто используются для стерилизации фруктов и овощей и лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в свою систему автономного вождения, а Waymo даже запустила приложение для Android для владельцев беспилотных транспортных средств.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон»,

# арабский

In [12]:
df_pred = await main('ar', 'арабский')

Аугментируем 120 текстов...
Переведено на арабский
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38-3:54 На самом деле радиация сама по себе не заразна. Например, мощные дозы радиации одних и тех же изотопов часто используются для стерилизации фруктов и овощей, а также для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в свою автоматизированную систему вождения, а Waymo запустила приложение для Android для владельцев беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин со

# африкаанс

In [13]:
df_pred = await main('af', 'африкаанс')

Аугментируем 120 текстов...
Переведено на африкаанс
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38–3:54 На самом деле радиация сама по себе не заразна. Например, мощные дозы радиации одних и тех же изотопов часто используются для стерилизации фруктов и овощей, а также для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в собственную систему беспилотного вождения, а Waymo даже запустила приложение для Android для владельцев беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядю

# зулу

In [14]:
df_pred = await main('zu', 'зулу')

Аугментируем 120 текстов...
Переведено на зулу
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38–3:54 На самом деле радиация сама по себе не заразна. Например, высокие дозы радиации одних и тех же изотопов часто используются для стерилизации фруктов и овощей, а также для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в свою систему беспилотного вождения, а Waymo даже запустила приложение для Android для владельцев беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», 

# латинский

In [15]:
df_pred = await main('la', 'латинский')

Аугментируем 120 текстов...
Переведено на латинский
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38-3:54 Радиация сама по себе не является инфекцией. Например, мощные дозы радиации тех же изотопов часто используются для стерилизации яблок и овощей, а также для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в свою систему беспилотных автомобилей, а Waymo также запустила приложение для Android для владельцев своих беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», 

# греческий

In [16]:
df_pred = await main('el', 'греческий')

Аугментируем 120 текстов...
Переведено на греческий
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38–3:54 На самом деле радиация сама по себе не заразна. Например, сильные дозы радиации одних и тех же изотопов часто используются для стерилизации фруктов и овощей, а также для лечения людей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в 1 миллиард долларов в собственную систему беспилотного вождения, а Waymo даже выпустила приложение для Android для владельцев беспилотных автомобилей.

2. Реальный текст:
	Множество повестей: «Двойник», «Дяд